In [23]:
from datetime import time
import re


def time_to_seconds(t: time) -> int:
    return t.hour * 3600 + t.minute * 60 + t.second


def seconds_to_time(s: int) -> time:
    h, remainder = divmod(s, 3600)
    m, s = divmod(remainder, 60)
    return time(h % 24, m, s)


def convert_to_24_hour_time(time_to_normalize: str) -> time:
    match: re.Match[str] | None = re.match(
        r"(\d{2}):(\d{2}):(\d{2})", time_to_normalize)
    if not match:
        raise ValueError(f"Invalid time format: {time_to_normalize}")

    hour, minute, second = map(int, match.groups())

    if hour >= 24:
        hour -= 24

    return time(hour, minute, second)


In [24]:
from dataclasses import dataclass
from datetime import time
from geopy.point import Point
from collections import defaultdict
import sys
import enum


@dataclass
class Node:
    name: str
    location: Point
    _hash: int = 0

    def __post_init__(self) -> None:
        self._hash = hash((self.name, self.location.format_unicode()))

    def __hash__(self) -> int:
        return self._hash


@dataclass
class CommunicationStep:
    company: str
    line: str
    departure_time: time
    arrival_time: time
    start_stop: Node
    end_stop: Node

    @staticmethod
    def from_parsed_csv_line(row: list[str]) -> "CommunicationStep":
        start_stop_point: Point = Point(latitude=row[7], longitude=row[8])
        start_stop = Node(row[5], start_stop_point)

        end_stop_point: Point = Point(latitude=row[9], longitude=row[10])
        end_stop = Node(row[6], end_stop_point)

        company, line, departure_str, arrival_str = row[1:5]

        departure_time: time = convert_to_24_hour_time(departure_str)
        arrival_time: time = convert_to_24_hour_time(arrival_str)

        new_communication_step = CommunicationStep(
            company, line, departure_time, arrival_time, start_stop, end_stop)

        return new_communication_step

    def __str__(self):
        return f"line {self.line} | {self.start_stop} {self.departure_time} -> {self.end_stop} {self.arrival_time}"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, CommunicationStep):
            return False
        return (
            self.company == other.company and
            self.line == other.line and
            self.departure_time == other.departure_time and
            self.arrival_time == other.arrival_time and
            self.start_stop == other.start_stop and
            self.end_stop == other.end_stop
        )

    def __hash__(self) -> int:
        return hash((self.company, self.line, self.departure_time, self.arrival_time, self.start_stop, self.end_stop))


class Graph:
    def __init__(self) -> None:
        self.nodes: dict[str, Node] = {}
        self.edges: dict[tuple[str, str],
                         set[CommunicationStep]] = defaultdict(set)
        self.adjacency_list: dict[str,
                                  set[CommunicationStep]] = defaultdict(set)


@dataclass
class LineStep:
    start_node_name: str
    end_node_name: str
    line: str
    start_time: time
    end_time: time


@dataclass
class Path:
    steps: list[LineStep]
    cost: float
    calculation_time: float

    def pretty_print(self) -> None:
        print("Schedule:")
        for step in self.steps:
            print(
                f"{step.line}\t| {step.start_node_name} {step.start_time} -> {step.end_node_name} {step.end_time}")
        print(f"Total cost: {self.cost} units", file=sys.stderr, flush=True)
        print(
            f"Execution time: {self.calculation_time:.4f} seconds", file=sys.stderr, flush=True)


class NoPathFoundError(Exception):
    """Raised when no path is found between the start and end nodes."""
    pass


class OptimizationCriterion(enum.Enum):
    TIME = "time"
    TRANSFERS = "transfers"


In [25]:
from geopy.distance import geodesic
from functools import lru_cache
from datetime import time


@lru_cache(maxsize=None)
def distance_heuristic(node1: Node, node2: Node) -> float:
    return geodesic(node1.location, node2.location).kilometers ** 2


def transfer_heuristic(transfer_count: int) -> float:
    transfer_penalty_weight = 500000
    return transfer_count * transfer_penalty_weight


def clear_caches() -> None:
    distance_heuristic.cache_clear()


def post_clear_cache(func):
    def wrapper(*args, **kwargs) -> None:
        result = func(*args, **kwargs)
        clear_caches()
        return result
    return wrapper


def generate_path(start: str, path_taken: list[CommunicationStep], elapsed: float, cost: float) -> Path:
    path_steps: list[LineStep] = []

    none_time: time = time(0, 0)

    step: CommunicationStep = path_taken[0]
    last_line: str = step.line
    path_steps.append(LineStep(
        start, "", step.line, step.departure_time, none_time))

    for i, step in enumerate(path_taken[:-1]):
        if step.line == last_line:
            continue
        last_line = step.line

        path_steps[-1].end_time = path_taken[i - 1].arrival_time
        path_steps[-1].end_node_name = step.start_stop.name

        path_steps.append(LineStep(step.start_stop.name,
                          "", last_line, step.departure_time, none_time))

    step = path_taken[-1]
    path_steps[-1].end_time = step.arrival_time
    path_steps[-1].end_node_name = step.end_stop.name

    path: Path = Path(path_steps, cost, elapsed)
    return path


In [26]:
from dataclasses import dataclass, field
from datetime import time
from datetime import datetime
import heapq
from itertools import count
import time as t


@dataclass(order=True)
class QueueEntry:
    priority: int
    counter: int
    current_stop_name: str = field(compare=False)
    path_taken: list[CommunicationStep] = field(compare=False)
    current_time_sec: int = field(compare=False)


@post_clear_cache
def dijkstra(start: str, end: str, start_time: time, graph: Graph) -> Path:
    if start not in graph.nodes:
        raise ValueError("Start stop does not exist in the graph.")
    if end not in graph.nodes:
        raise ValueError("End stop does not exist in the graph.")

    start_node: Node = graph.nodes[start]
    end_node: Node = graph.nodes[end]

    start_time_sec: int = time_to_seconds(start_time)

    queue: list[QueueEntry] = []
    counter = count()
    heapq.heappush(queue, QueueEntry(0, next(counter),
                   start_node.name, [], start_time_sec))

    visited: set[str] = set()

    start_time_perf: float = t.perf_counter()

    while queue:
        entry: QueueEntry = heapq.heappop(queue)

        if entry.current_stop_name in visited:
            continue
        visited.add(entry.current_stop_name)

        if entry.current_stop_name == end_node.name:
            elapsed: float = t.perf_counter() - start_time_perf

            path: Path = generate_path(
                start, entry.path_taken, elapsed, entry.priority)

            return path

        for (start_name, end_name), steps in graph.edges.items():
            if start_name != entry.current_stop_name:
                continue
            for step in steps:
                departure_seconds: int = time_to_seconds(step.departure_time)
                arrival_seconds: int = time_to_seconds(step.arrival_time)

                if departure_seconds >= entry.current_time_sec:
                    travel_time: int = arrival_seconds - entry.current_time_sec
                    if travel_time < 0:
                        travel_time += 86400  # handle crossing midnight

                    new_priority: int = entry.priority + travel_time
                    new_queue_entry = QueueEntry(
                        new_priority,
                        next(counter),
                        end_name,
                        entry.path_taken + [step],
                        arrival_seconds
                    )
                    heapq.heappush(queue, new_queue_entry)

    raise NoPathFoundError(f"No path found from {start} to {end}")


In [27]:
import pandas as pd
import pickle
import os

csv_filename = "connection_graph"
separator = ","
skipfooter = 0
graph: Graph


def try_load_from_pickle() -> Graph | None:
    graph: Graph | None = None

    if os.path.exists(f"data/{csv_filename}.pkl"):
        with open(f"data/{csv_filename}.pkl", "rb") as f:
            graph = pickle.load(f)
        print("Graph loaded from graph.pkl.")
    if os.path.exists(f"lab01/data/{csv_filename}.pkl"):
        with open(f"lab01/data/{csv_filename}.pkl", "rb") as f:
            graph = pickle.load(f)
        print("Graph loaded from graph.pkl.")

    return graph


def load_from_csv() -> Graph:
    graph: Graph

    df: pd.DataFrame
    if os.path.exists(f"data/{csv_filename}.csv"):
        df = pd.read_csv(f"data/{csv_filename}.csv", encoding="utf-8",
                         sep=separator, skipfooter=skipfooter, engine="python")
    else:
        df = pd.read_csv(f"lab01/data/{csv_filename}.csv", encoding="utf-8",
                         sep=separator, skipfooter=skipfooter, engine="python")

    graph = Graph()

    for _, row in df.iterrows():
        step: CommunicationStep = CommunicationStep.from_parsed_csv_line(
            list(row))

        for stop in [step.start_stop, step.end_stop]:
            if stop.name not in graph.nodes:
                graph.nodes[stop.name] = stop
                graph.adjacency_list[stop.name] = set()

        key: tuple[str, str] = (step.start_stop.name, step.end_stop.name)
        if step not in graph.edges[key]:
            graph.edges[key].add(step)
            graph.adjacency_list[step.start_stop.name].add(step)

    with open(f"data/{csv_filename}.pkl", "wb") as f2:
        pickle.dump(graph, f2)

    print("Done parsing .csv")

    return graph


def load_from_pickle_fallback_csv() -> Graph:
    graph: Graph | None = try_load_from_pickle()

    if graph is not None:
        return graph

    graph = load_from_csv()

    return graph


In [28]:
graph: Graph = load_from_pickle_fallback_csv()

Graph loaded from graph.pkl.


In [29]:
a = "Prusa"
b = "PORT LOTNICZY"
optimization_criterion_str = "p"
start_time = "08:00:00"

start_time_obj: time = datetime.strptime(start_time, "%H:%M:%S").time()

path: Path = dijkstra(a, b, start_time_obj, graph)
path.pretty_print()

Total cost: 3420 units
Execution time: 5.4618 seconds


Schedule:
1	| Prusa 08:02:00 -> Wyszyńskiego 08:03:00
128 	| Wyszyńskiego 08:03:00 -> Dubois 08:09:00
6	| Dubois 08:10:00 -> Rynek 08:15:00
12	| Rynek 08:15:00 -> PL. JANA PAWŁA II 08:17:00
122	| PL. JANA PAWŁA II 08:17:00 -> pl. Orląt Lwowskich 08:19:00
148	| pl. Orląt Lwowskich 08:20:00 -> Strzegomska (krzyżówka) 08:32:00
134	| Strzegomska (krzyżówka) 08:32:00 -> Rogowska (P+R) 08:34:00
132	| Rogowska (P+R) 08:35:00 -> MIŃSKA (Rondo Rotm. Pileckiego) 08:37:00
106	| MIŃSKA (Rondo Rotm. Pileckiego) 08:43:00 -> PORT LOTNICZY 08:57:00


In [30]:
from dataclasses import dataclass, field
from datetime import time
import heapq
from itertools import count
import time as t


@dataclass(order=True)
class QueueEntry:
    priority: float
    counter: int
    current_stop_name: str = field(compare=False)
    path_taken: list[CommunicationStep] = field(compare=False)
    current_time_sec: int = field(compare=False)
    transfer_count: int = field(compare=False)


def should_skip(entry: QueueEntry, best_state: dict[str, tuple[int, int]]) -> bool:
    current_state: tuple[int, int] | None = best_state.get(
        entry.current_stop_name)
    if current_state is None:
        return False
    best_transfers, best_arrival = current_state
    return (entry.transfer_count > best_transfers) or \
           (entry.transfer_count ==
            best_transfers and entry.current_time_sec >= best_arrival)


def update_best_state(entry: QueueEntry, best_state: dict[str, tuple[int, int]]) -> None:
    best_state[entry.current_stop_name] = (
        entry.transfer_count, entry.current_time_sec)


def calculate_priority(entry: QueueEntry, current_node: Node, neighbor_node: Node, step: CommunicationStep,
                       travel_time: int, optimization_criterion: OptimizationCriterion) -> tuple[float, int]:
    cost_so_far: float = entry.priority + travel_time
    estimated_remaining: float = 0
    new_transfer_count: int = entry.transfer_count

    if optimization_criterion == OptimizationCriterion.TIME:
        cost_so_far -= distance_heuristic(current_node, neighbor_node)
        estimated_remaining = distance_heuristic(neighbor_node, neighbor_node)

    if optimization_criterion == OptimizationCriterion.TRANSFERS:
        cost_so_far += transfer_heuristic(entry.transfer_count)
        if entry.path_taken:
            previous_step: CommunicationStep = entry.path_taken[-1]
            is_transfer: bool = previous_step.line != step.line
            new_transfer_count += 1 if is_transfer else 0

    total_priority: float = cost_so_far + estimated_remaining
    return total_priority, new_transfer_count


@post_clear_cache
def a_star_search(start: str, end: str, start_time: time, graph: Graph, optimization_criterion: OptimizationCriterion) -> Path:
    if start not in graph.nodes or end not in graph.nodes:
        raise ValueError("Start or end stop does not exist in the graph.")

    start_node: Node = graph.nodes[start]
    end_node: Node = graph.nodes[end]
    start_time_sec: int = time_to_seconds(start_time)

    queue: list[QueueEntry] = []
    counter = count()
    initial_heuristic: float = distance_heuristic(
        start_node, end_node) if optimization_criterion == OptimizationCriterion.TIME else 0

    heapq.heappush(queue, QueueEntry(initial_heuristic, next(
        counter), start_node.name, [], start_time_sec, 0))
    best_state: dict[str, tuple[int, int]] = {}
    start_time_perf: float = t.perf_counter()

    while queue:
        entry: QueueEntry = heapq.heappop(queue)
        if should_skip(entry, best_state):
            continue
        update_best_state(entry, best_state)
        if entry.current_stop_name == end_node.name:
            elapsed: float = t.perf_counter() - start_time_perf
            return generate_path(start, entry.path_taken, elapsed, entry.priority)

        current_node: Node = graph.nodes[entry.current_stop_name]
        for (start_name, end_name), steps in graph.edges.items():
            if start_name != entry.current_stop_name:
                continue
            for step in steps:
                departure_seconds: int = time_to_seconds(step.departure_time)
                arrival_seconds: int = time_to_seconds(step.arrival_time)

                if departure_seconds >= entry.current_time_sec:
                    # handle midnight wrap
                    travel_time: int = (
                        arrival_seconds - entry.current_time_sec) % 86400

                    neighbor_node: Node = graph.nodes[end_name]
                    total_priority, new_transfer_count = calculate_priority(
                        entry, current_node, neighbor_node, step, travel_time, optimization_criterion
                    )
                    new_entry = QueueEntry(
                        total_priority,
                        next(counter),
                        end_name,
                        entry.path_taken + [step],
                        arrival_seconds,
                        new_transfer_count
                    )
                    heapq.heappush(queue, new_entry)

    raise NoPathFoundError(f"No path found from {start} to {end}")


In [31]:
a = "Prusa"
b = "PORT LOTNICZY"
optimization_criterion_str = "p"
start_time = "08:00:00"

start_time_obj: time = datetime.strptime(start_time, "%H:%M:%S").time()
optimization_criterion: OptimizationCriterion = OptimizationCriterion.TIME

path: Path = a_star_search(a, b, start_time_obj, graph, optimization_criterion)
path.pretty_print()

optimization_criterion: OptimizationCriterion = OptimizationCriterion.TRANSFERS
path: Path = a_star_search(a, b, start_time_obj, graph, optimization_criterion)
path.pretty_print()

Total cost: 3564.9354059495113 units
Execution time: 1.5706 seconds


Schedule:
1	| Prusa 08:02:00 -> Wyszyńskiego 08:03:00
128 	| Wyszyńskiego 08:03:00 -> Dubois 08:09:00
6	| Dubois 08:10:00 -> Rynek 08:15:00
12	| Rynek 08:15:00 -> PL. JANA PAWŁA II 08:17:00
122	| PL. JANA PAWŁA II 08:17:00 -> pl. Orląt Lwowskich 08:19:00
148	| pl. Orląt Lwowskich 08:20:00 -> Strzegomska (krzyżówka) 08:32:00
134	| Strzegomska (krzyżówka) 08:32:00 -> Rogowska (P+R) 08:34:00
132	| Rogowska (P+R) 08:35:00 -> MIŃSKA (Rondo Rotm. Pileckiego) 08:37:00
106	| MIŃSKA (Rondo Rotm. Pileckiego) 08:43:00 -> PORT LOTNICZY 08:57:00


Total cost: 22503720 units
Execution time: 10.2711 seconds


Schedule:
1	| Prusa 08:02:00 -> Wyszyńskiego 08:03:00
128 	| Wyszyńskiego 08:03:00 -> KUŹNIKI 08:33:00
129	| KUŹNIKI 08:41:00 -> PORT LOTNICZY 09:02:00


In [32]:
from random import randint
from abc import ABC, abstractmethod


class TabuSizeStrategy(ABC):
    @abstractmethod
    def get_tabu_size(self, required_stops: list[str]) -> int:
        ...


class FixedTabuSizeStrategy(TabuSizeStrategy):
    def __init__(self, size: int = 10) -> None:
        self.size: int = size

    def get_tabu_size(self, required_stops: list[str]) -> int:
        return self.size


class DynamicTabuSizeStrategy(TabuSizeStrategy):
    def __init__(self, k: float = 5.0, min_size: int = 10) -> None:
        self.k: float = k
        self.min_size: int = min_size

    def get_tabu_size(self, required_stops: list[str]) -> int:
        return max(self.min_size, int(self.k * len(required_stops)))


class NeighborhoodSamplingStrategy(ABC):
    @abstractmethod
    def generate_swaps(self, num_stops: int) -> list[tuple[int, int]]:
        ...


class FullSamplingStrategy(NeighborhoodSamplingStrategy):
    def generate_swaps(self, num_stops: int) -> list[tuple[int, int]]:
        swaps: list[tuple[int, int]] = []
        for i in range(1, num_stops):
            for j in range(i+1, num_stops+1):
                swaps.append((i, j))
        return swaps


class RandomSamplingStrategy(NeighborhoodSamplingStrategy):
    def __init__(self, sample_size: int = 10) -> None:
        self.sample_size: int = sample_size

    def generate_swaps(self, num_stops: int) -> list[tuple[int, int]]:
        swaps: set[tuple[int, int]] = set()
        while len(swaps) < min(self.sample_size, (num_stops * (num_stops - 1)) // 2):
            i: int = randint(1, num_stops - 1)
            j: int = randint(i+1, num_stops)
            swaps.add((i, j))
        return list(swaps)


class AspirationStrategy(ABC):
    @abstractmethod
    def allow_route(self, tabu_set: set[tuple[str]], route: tuple[str, ...]) -> bool:
        ...


class StrictTabuAspirationStrategy(AspirationStrategy):
    def allow_route(self, tabu_set, route) -> bool:
        return route not in tabu_set


class AllowTabuAspirationStrategy(AspirationStrategy):
    def allow_route(self, tabu_set, route) -> bool:
        return True


In [33]:
from collections import deque
from dataclasses import dataclass, field
from utils import time_to_seconds
from datetime import datetime, time, timedelta
import time as t
from models import NoPathFoundError, OptimizationCriterion, Path, CommunicationStep, Graph, LineStep, Node
from a_star import a_star_search
from functools import lru_cache
from tabu_strategies import AspirationStrategy, AllowTabuAspirationStrategy, StrictTabuAspirationStrategy, FixedTabuSizeStrategy, TabuSizeStrategy, FullSamplingStrategy, NeighborhoodSamplingStrategy
import heapq
from collections import deque
from geopy.distance import geodesic


@lru_cache(maxsize=None)
def get_shortest_path(start: str, end: str, start_time_sec: int, graph: Graph, optimization_criterion: OptimizationCriterion) -> Path:
    start_time_obj: time = (
        datetime.min + timedelta(seconds=start_time_sec)).time()
    path: Path = a_star_search(
        start, end, start_time_obj, graph, optimization_criterion)
    return path


@dataclass(order=True)
class TabuSolution:
    cost: int
    route: list[str] = field(compare=False)
    steps: list[CommunicationStep] = field(compare=False)


def post_clear_cache(func):
    def wrapper(*args, **kwargs) -> None:
        result = func(*args, **kwargs)
        get_shortest_path.cache_clear()
        return result
    return wrapper


def calculate_total_cost_and_steps(route: list[str], start_time: time, graph: Graph, optimization_criterion: OptimizationCriterion):
    total_cost: float = 0
    steps: list[LineStep] = []
    current_time: time = start_time

    for i in range(len(route) - 1):
        start: str = route[i]
        end: str = route[i+1]

        start_time_sec: int = time_to_seconds(current_time)
        try:
            path: Path = get_shortest_path(
                start, end, start_time_sec, graph, optimization_criterion)
        except NoPathFoundError:
            return float("inf"), None
        total_cost += path.cost
        steps += path.steps

        current_time = path.steps[-1].end_time if path.steps else current_time

    return total_cost, steps


def estimate_good_first_path(start: str, route: list[str], graph: Graph) -> list[str]:
    nodes: list[Node] = [graph.nodes[stop] for stop in route]

    start_node: Node = graph.nodes[start]

    current_stop: Node = start_node

    path: list[str] = [start]

    while len(nodes) > 0:
        nodes.sort(key=lambda node: geodesic(
            current_stop.location, node.location).km)

        path.append(nodes[0].name)
        current_stop = nodes.pop(0)

    path.append(start)

    return path


@post_clear_cache
def tabu_search(start: str, required_stops: list[str], start_time: time, graph: Graph,
                optimization_criterion: OptimizationCriterion, max_iterations=5,
                tabu_size_strategy: TabuSizeStrategy = FixedTabuSizeStrategy(),
                sampling_strategy: NeighborhoodSamplingStrategy = FullSamplingStrategy(),
                aspiration_strategy: AspirationStrategy = StrictTabuAspirationStrategy(),
                ) -> Path:

    current_route: list[str] = estimate_good_first_path(
        start, required_stops, graph)
    best_route: list[str] = current_route.copy()

    best_cost, best_steps = calculate_total_cost_and_steps(
        best_route, start_time, graph, optimization_criterion)

    tabu_size: int = tabu_size_strategy.get_tabu_size(required_stops)

    tabu_set: set[tuple] = set()
    tabu_queue: deque[tuple] = deque()

    start_time_perf: float = t.perf_counter()

    for _ in range(max_iterations):
        neighborhood: list[TabuSolution] = []
        swaps: list[tuple[int, int]] = sampling_strategy.generate_swaps(
            len(required_stops))

        for i, j in swaps:
            new_route: list[str] = current_route.copy()
            new_route[i], new_route[j] = new_route[j], new_route[i]
            route_tuple: tuple[str, ...] = tuple(new_route)

            if not aspiration_strategy.allow_route(tabu_set, route_tuple):
                continue

            cost, steps = calculate_total_cost_and_steps(
                new_route, start_time, graph, optimization_criterion)

            if (route_tuple not in tabu_set) or (cost < best_cost):
                heapq.heappush(neighborhood, TabuSolution(
                    cost, new_route, steps))

        if not neighborhood:
            break

        best_neighbor: TabuSolution = heapq.heappop(neighborhood)

        if best_neighbor.cost < best_cost:
            best_cost = best_neighbor.cost
            best_route = best_neighbor.route
            best_steps = best_neighbor.steps

        current_route = best_neighbor.route
        route_tuple = tuple(current_route)
        tabu_set.add(route_tuple)
        tabu_queue.append(route_tuple)

        if len(tabu_queue) > tabu_size:
            oldest = tabu_queue.popleft()
            tabu_set.remove(oldest)

    elapsed: float = t.perf_counter() - start_time_perf

    return Path(best_steps, best_cost, elapsed)


In [ ]:
stops: list[str] = ["C.H. Korona", "FAT", "GAJ"]

print("Tabu")
path: Path = tabu_search(a, stops, start_time_obj, graph, OptimizationCriterion.TIME,
                    sampling_strategy=RandomSamplingStrategy(3))
path.pretty_print()

print("Tabu 2")
path = tabu_search(a, stops, start_time_obj, graph,
                    OptimizationCriterion.TRANSFERS, sampling_strategy=RandomSamplingStrategy(3))
path.pretty_print()

print("Tabu 3")
path = tabu_search(a, stops, start_time_obj, graph, OptimizationCriterion.TIME,
                    tabu_size_strategy=DynamicTabuSizeStrategy(k=5.0, min_size=10), sampling_strategy=RandomSamplingStrategy(3))
path.pretty_print()

print("Tabu 4")
path = tabu_search(a, stops, start_time_obj, graph,
                    OptimizationCriterion.TRANSFERS, tabu_size_strategy=FixedTabuSizeStrategy(sys.maxsize), sampling_strategy=RandomSamplingStrategy(3))
path.pretty_print()

Tabu


AttributeError: 'str' object has no attribute 'hour'